# Edge S2 feature pass (embeddings, hashes, SQ-GAN, decode, timings) + SPI grid

Step `edge-s2`. Thin wrapper; logic lives in `Codec.edge.feature_pass`.


In [ ]:
import os, subprocess
REPO, URL, BRANCH = "/kaggle/working/repo", "https://github.com/saltypal/SemanticSchedulerEnd2End.git", "main"
if os.path.isdir(REPO + "/.git"):
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", "origin/" + BRANCH], check=True)
else:
    subprocess.run(f"git clone --depth 1 --branch {BRANCH} {URL} {REPO}", shell=True, check=True)
GIT_SHA = subprocess.check_output(f"git -C {REPO} rev-parse HEAD", shell=True, text=True).strip()
print("commit", GIT_SHA)

In [ ]:
%cd /kaggle/working/repo
!pip install -q -e . lpips

In [ ]:
import time; T0 = time.time()

In [ ]:
!python -u -m Codec.edge.feature_pass run --out /kaggle/working/runs/edge-s2

## Compute profile (once per model)


In [ ]:
!python -u -m Codec.edge.compute_profile --out /kaggle/working/runs/edge-s2

## Grid hand-off

Provisional splits are built by `parameters.edge.splits.assign_event_splits` when `splits_v1.json` is absent from the labels dataset; the export step prints which was used.


In [ ]:
!python -u -m Codec.edge.feature_pass export-grid --out /kaggle/working/runs/edge-s2

In [ ]:
!python -u -m parameters.edge.embedding_grid grid --embeddings-dir /kaggle/working/runs/edge-s2/grid --labels /kaggle/working/runs/edge-s2/grid/labels_by_uid.jsonl --splits /kaggle/working/runs/edge-s2/grid/splits_by_uid.json --out /kaggle/working/runs/edge-s2/grid

In [ ]:
!python -u -m parameters.edge.embedding_grid select --grid-dir /kaggle/working/runs/edge-s2/grid

In [ ]:
import datetime, json, os, platform, subprocess, time
STEP, ARGS = "edge-s2", ['run', '--out', '/kaggle/working/runs/edge-s2']
def _gpu():
    try:
        return subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
    except Exception:
        return None
try:
    import torch; torch_version = torch.__version__
except Exception:
    torch_version = None
out = "/kaggle/working/runs/" + STEP
os.makedirs(out, exist_ok=True)
manifest = {"step": STEP, "entrypoint": "Codec.edge.feature_pass", "args": ARGS, "git_sha": GIT_SHA,
            "utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "python": platform.python_version(),
            "torch": torch_version, "gpu": _gpu(), "wall_time_s": time.time() - T0}
json.dump(manifest, open(out + "/run_manifest.json", "w"), indent=2)
print(manifest)

## Publish cache (dataset scheduler-dataset-v1-cache)

Skipped with a reason if no credentials.


In [ ]:
!python -u -m Codec.edge.feature_pass publish --out /kaggle/working/runs/edge-s2 --slug scheduler-dataset-v1-cache